### Example from GPT
* Try wrap with LangChain for the querying part

In [ ]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from PIL import Image
import requests
from io import BytesIO

# Init Chroma client (can also set persist_directory for disk storage)
client = chromadb.Client(Settings(anonymized_telemetry=False))
collection = client.get_or_create_collection(name="crystal-drops")

# Load embedding model
model = SentenceTransformer("clip-ViT-B-32")

# Sample image + caption data
samples = [
    {
        "id": "drop1",
        "image_url": "https://marco-dataset.s3.us-east-2.amazonaws.com/positive/10044_2_2819465_1_1.jpg",
        "caption": "Drop contains small needle-like crystals scattered across the well."
    },
    {
        "id": "drop2",
        "image_url": "https://marco-dataset.s3.us-east-2.amazonaws.com/negative/10053_2_2853662_1_1.jpg",
        "caption": "Amorphous precipitate dominates the entire drop with no clear crystal structures."
    }
]

# Embed and add to Chroma
for s in samples:
    embedding = model.encode(s["caption"])
    collection.add(
        documents=[s["caption"]],
        metadatas=[{"image_url": s["image_url"]}],
        ids=[s["id"]],
        embeddings=[embedding.tolist()]
    )

# Step 2: Query the collection
query = "What does a drop with needle-shaped crystals look like?"
query_embedding = model.encode(query).tolist()

results = collection.query(query_embeddings=[query_embedding], n_results=2)

# Step 3: Construct RAG prompt
context_snippets = "\n".join(f"- {doc}" for doc in results['documents'][0])
prompt = f"""You are assisting in analyzing protein crystallization drops based on microscopy images.

Here are some previous examples:
{context_snippets}

Now, given the new description: "{query}", what is your interpretation?
Answer concisely using language similar to the examples above.
"""

print(prompt)
